# Group 6 PatchCore 小缺陷定位：Colab 完整复现 + 扩展指标

本 notebook 在 Google Colab 上重复本项目的全部实验，并用新指标检验原报告的结论。

| 部分 | 内容 |
|---|---|
| 主实验（54 组） | 3 类（screw / capsule / metal_nut）× 3 种特征（layer2 / layer3 / fusion）× 3 个 coreset 种子 × 2 种输入尺寸（224 / 320），coreset 比例 10% |
| 固定记忆库实验（27 组） | 224 像素，每个记忆库 4096 个向量（16 MiB） |
| 原报告的分析 | 数据审计、三种子均值/标准差、layer2 vs fusion 配对 bootstrap、示例热力图 |
| **新增：扩展指标** | 从保存的预测结果事后计算（不重跑模型），检查"layer2 ≥ fusion""layer2 > layer3""320 > 224"在新指标下是否仍成立 |

**使用前请注意**

1. 菜单「代码执行程序 → 更改运行时类型」选 **GPU**（T4 即可，L4 / A100 更快）。
2. Google Drive 至少留出约 **10 GB**：81 组运行的完整输出约 7.8 GB，另有数据压缩包和汇总结果。
3. 实验结果直接写入 Drive。**断线后从第 1 步起依次重新运行各单元格即可续跑**：已完成的运行会被跳过，没跑完的运行会先被移到 `incomplete/` 目录。长时间运行时请保持浏览器页面打开。
4. Colab 的 GPU / CPU 与原实验的 Windows RTX 5060 笔记本不同：**精度指标可以和原报告对照，延迟不能对照。**
5. 本 notebook 不修改 `study/` 和 `run_experiment.py`；新代码写在单独的 `extended_metrics.py` 中，所以源码哈希应与原 Windows 实验一致，第 2 步会自动核对。

## 1. 挂载 Google Drive 并设置路径

所有需要跨会话保存的内容（数据压缩包、实验结果、日志、汇总）都放在 `DRIVE_ROOT` 下。数据解压到 Colab 本地磁盘，读图更快。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import hashlib, json, os, pathlib, platform, shutil, subprocess, sys, time

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/group6_patchcore')  # 可按需修改
REPO_URL = 'https://github.com/bernie90103/CV_final_project.git'
REPO_DIR = pathlib.Path('/content/CV_final_project')  # 如修改，第 4 步 %%writefile 的路径也要一起改
PATCHCORE_URL = 'https://github.com/amazon-science/patchcore-inspection.git'
PATCHCORE_COMMIT = 'fcaa92f124fb1ad74a7acf56726decd4b27cbcad'  # THIRD_PARTY.md 中固定的提交
EXPECTED_SOURCE_SHA256 = '41cca92910a8c2e5610bb5abcd07786027a3decffaf9d117fbe2e766e201dfb4'  # 原 Windows 实验的 study_source_sha256

DATA_DIR = pathlib.Path('/content/data/mvtec')   # Colab 本地磁盘
ARCHIVE_DIR = DRIVE_ROOT / 'archives'            # 数据压缩包，避免重复下载
RUNS_DIR = DRIVE_ROOT / 'runs_colab'             # 81 组运行的完整输出
OUT_DIR = DRIVE_ROOT / 'outputs'                 # 审计、汇总、对比、扩展指标
LOG_DIR = OUT_DIR / 'logs'
for d in (ARCHIVE_DIR, RUNS_DIR, OUT_DIR, LOG_DIR):
    d.mkdir(parents=True, exist_ok=True)
PY = sys.executable


def sh(cmd, cwd=None, log=None, check=True):
    # 运行命令并实时打印输出；log 不为空时同时追加写入日志文件
    shown = cmd if isinstance(cmd, str) else ' '.join(map(str, cmd))
    print('$', shown, flush=True)
    proc = subprocess.Popen(cmd if isinstance(cmd, str) else [str(c) for c in cmd],
                            shell=isinstance(cmd, str), cwd=cwd, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    handle = open(log, 'a') if log else None
    try:
        if handle:
            handle.write(f'\n$ {shown}\n')
        for line in proc.stdout:
            print(line, end='', flush=True)
            if handle:
                handle.write(line)
    finally:
        if handle:
            handle.close()
    code = proc.wait()
    if check and code:
        raise RuntimeError(f'命令失败（退出码 {code}）：{shown}')
    return code

print('Drive 目录：', DRIVE_ROOT)

## 2. 下载项目代码与 PatchCore

克隆公开仓库及其 PatchCore 子模块。如果子模块没有拉下来，就单独克隆官方 PatchCore 并切到报告中固定的提交。`run_experiment.py` 会用 `git rev-parse` 记录 PatchCore 的版本，所以 `vendor/patchcore-inspection` 必须是一个 git 仓库。

In [ ]:
if not (REPO_DIR / '.git').exists():
    sh(['git', 'clone', '--recurse-submodules', REPO_URL, REPO_DIR])
else:
    print('项目代码已存在，跳过克隆：', REPO_DIR)

VENDOR = REPO_DIR / 'vendor/patchcore-inspection'
if not (VENDOR / '.git').exists():
    sh(['git', 'submodule', 'update', '--init', '--recursive'], cwd=REPO_DIR, check=False)
if not (VENDOR / '.git').exists():
    shutil.rmtree(VENDOR, ignore_errors=True)
    sh(['git', 'clone', PATCHCORE_URL, VENDOR])
head = subprocess.check_output(['git', '-C', str(VENDOR), 'rev-parse', 'HEAD'], text=True).strip()
if head != PATCHCORE_COMMIT:
    sh(['git', '-C', VENDOR, 'fetch', '--quiet', 'origin', PATCHCORE_COMMIT], check=False)
    sh(['git', '-C', VENDOR, 'checkout', '--quiet', PATCHCORE_COMMIT])
    head = subprocess.check_output(['git', '-C', str(VENDOR), 'rev-parse', 'HEAD'], text=True).strip()
assert head == PATCHCORE_COMMIT, f'PatchCore 提交号不符：{head}'
print('PatchCore 提交号正确：', head)


def source_fingerprint(root):
    # 与 run_experiment.source_fingerprint 相同：只哈希 run_experiment.py 和 study/*.py
    digest = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
    paths = [root / 'run_experiment.py', *sorted((root / 'study').glob('*.py'))]
    return hashlib.sha256(''.join(digest(p) for p in paths).encode()).hexdigest()

SOURCE_SHA256 = source_fingerprint(REPO_DIR)
if SOURCE_SHA256 == EXPECTED_SOURCE_SHA256:
    print('实验源码哈希与原 Windows 实验一致 ✅')
else:
    print('⚠️ 实验源码哈希与原 Windows 实验不同：', SOURCE_SHA256)
    print('   仓库代码可能已更新。实验仍可运行，但与原报告对照时要注意代码差异。')

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

## 3. 检查 GPU 并安装依赖

Colab 已预装带 CUDA 的 PyTorch，这里**不重装 torch / torchvision**，只安装其余依赖。如果 pip 提示需要重启，请选「重启会话」，然后从第 1 步重新运行（再次安装会很快跳过）。

In [ ]:
sh('nvidia-smi')
requirements = [line.strip() for line in (REPO_DIR / 'requirements.txt').read_text().splitlines()
                if line.strip() and not line.startswith(('#', 'torch'))]
sh([PY, '-m', 'pip', 'install', '-q', *requirements])

In [ ]:
import importlib.metadata
import torch

assert torch.cuda.is_available(), '没有可用的 GPU：请在「代码执行程序 → 更改运行时类型」中选择 GPU'
print('PyTorch', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
if not torch.__version__.startswith('2.'):
    print('⚠️ requirements.txt 要求 torch>=2.5,<3，当前版本不在范围内，结果可能与原实验有差异。')

# run_experiment.py 把以下包版本和系统信息写进 config.json。
# 记录第一次会话的环境，之后的会话如有变化就提醒。
PACKAGES = ['torch', 'torchvision', 'timm', 'faiss-cpu', 'numpy', 'scipy', 'scikit-learn']
env_now = {'packages': {p: importlib.metadata.version(p) for p in PACKAGES},
           'system': platform.platform(), 'gpu': torch.cuda.get_device_name(0)}
ENV_FILE = DRIVE_ROOT / 'environment_first_session.json'
if not ENV_FILE.exists():
    ENV_FILE.write_text(json.dumps(env_now, indent=2))
    print('已记录本次环境：')
    print(json.dumps(env_now, indent=2))
else:
    env_first = json.loads(ENV_FILE.read_text())
    diffs = {k: (env_first['packages'].get(k), v) for k, v in env_now['packages'].items()
             if env_first['packages'].get(k) != v}
    if env_first['system'] != env_now['system']:
        diffs['system'] = (env_first['system'], env_now['system'])
    if env_first.get('gpu') != env_now['gpu']:
        diffs['gpu'] = (env_first.get('gpu'), env_now['gpu'])
    if diffs:
        print('⚠️ 与第一次会话的环境不同（第一次 → 现在）：')
        for k, (a, b) in diffs.items():
            print(f'   {k}: {a} → {b}')
        print('   已完成的运行仍会被跳过（本 notebook 按目录判断）。summarize_results.py 会按 system 分组，')
        print('   所以同一配置可能被拆成多行；第 14 步的对照和第 13 步的扩展指标不受影响。')
        print('   GPU 型号不同时，新跑的运行与旧运行的延迟不能放在一起比较。')
    else:
        print('环境与第一次会话一致 ✅')

## 4. 写入扩展指标模块并运行单元测试

`extended_metrics.py` 只读取每次运行保存的 `predictions.npz` 和 `samples.json`，事后计算新指标，不重跑模型。它放在 `study/` 之外，所以不会改变实验源码哈希。

In [ ]:
%%writefile /content/CV_final_project/extended_metrics.py
"""Post-hoc extended metrics for completed PatchCore runs (Colab add-on).

Reads only each run's saved predictions.npz and samples.json; no model is
rebuilt. This file lives outside study/ so study_source_sha256 is unchanged.

Thresholds never use defect masks, with one labelled exception: pixel F1-max
picks its best threshold on the test masks and is reported only as an oracle
diagnostic. Region recall uses a threshold taken from normal test images only.
Subgroups follow study.metrics.evaluate_groups: every subgroup contains all
normal test images plus its own anomalous images.
"""
import argparse
from collections import defaultdict
import json
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from sklearn.metrics import roc_curve

from study.io import write_csv, write_json
from study.metrics import pro_curve, safe_auroc

GROUPS = ('all', 'small', 'large')
FPR_LIMITS = (.3, .05, .01)
IMAGE_FPR = .05
SIZE_EDGES = (0, 64, 256, 1024)
SIZE_LABELS = ('<64', '64-255', '256-1023', '>=1024')
LAYER_ORDER = ('layer2', 'layer3', 'fusion')
STRUCTURE = np.ones((3, 3), dtype=bool)


def metric_names(tolerance=8, normal_fpr=.01):
    return ([f'aupro@{limit:g}' for limit in FPR_LIMITS]
            + ['pixel_ap', 'pixel_f1max_oracle', 'image_auroc', 'image_ap', f'image_tpr@fpr{IMAGE_FPR:g}',
               'pointing@0px', f'pointing@{tolerance:g}px', f'region_recall@nfpr{normal_fpr:g}'])


def size_bin(n_pixels):
    """Size class of one connected defect region, in transformed-mask pixels."""
    if n_pixels < 1:
        raise ValueError('A region has at least one pixel.')
    return SIZE_LABELS[int(np.searchsorted(SIZE_EDGES, n_pixels, side='right')) - 1]


def aupro_from_curve(curve, limit):
    """Same clipping and integration as study.metrics.aupro, reusing one PRO curve."""
    if curve is None:
        return None
    x, y = curve
    right = int(np.searchsorted(x, limit, side='right'))
    clipped_x, clipped_y = x[:right], y[:right]
    if clipped_x[-1] < limit:
        value = np.interp(limit, x[right-1:right+1], y[right-1:right+1])
        clipped_x, clipped_y = np.r_[clipped_x, limit], np.r_[clipped_y, value]
    return float(np.clip(np.trapezoid(clipped_y, clipped_x) / limit, 0, 1))


def ranking_ap_f1(labels, scores):
    """Step-wise average precision and best F1 over tied score thresholds.

    AP matches sklearn.metrics.average_precision_score. One sort serves both
    values, which keeps millions of pixels affordable.
    """
    labels = np.asarray(labels, dtype=bool).ravel()
    scores = np.asarray(scores, dtype=np.float64).ravel()
    if labels.shape != scores.shape:
        raise ValueError('Labels and scores must have the same size.')
    if not np.isfinite(scores).all():
        raise ValueError('Non-finite scores.')
    n_pos = int(labels.sum())
    if n_pos == 0 or n_pos == labels.size:
        return None, None
    order = np.argsort(-scores, kind='stable')
    ordered = scores[order]
    # A tied score enters the ranking only as one complete block.
    ends = np.r_[np.flatnonzero(ordered[:-1] != ordered[1:]), ordered.size - 1]
    tp = np.cumsum(labels[order], dtype=np.int64)[ends]
    precision = tp / (ends + 1)
    recall = tp / n_pos
    ap = float(np.sum(np.diff(np.r_[0.0, recall]) * precision))
    denominator = precision + recall
    f1 = np.divide(2 * precision * recall, denominator, out=np.zeros_like(denominator), where=denominator > 0)
    return ap, float(f1.max())


def tpr_at_fpr(labels, scores, fpr=IMAGE_FPR):
    """Best achievable TPR among operating points whose FPR does not exceed fpr."""
    labels = np.asarray(labels)
    if len(np.unique(labels)) != 2:
        return None
    fprs, tprs, _ = roc_curve(labels, scores, drop_intermediate=False)
    return float(tprs[fprs <= fpr + 1e-12].max())


def normal_pixel_threshold(rows, maps, fpr):
    """Pixel threshold from normal test images only; defect masks are not used."""
    if not 0 < fpr < 1:
        raise ValueError('normal_fpr must be in (0,1).')
    normal = np.array([not r['is_anomaly'] for r in rows])
    if not normal.any():
        raise ValueError('Region recall needs at least one normal test image.')
    return float(np.quantile(np.asarray(maps)[normal], 1 - fpr))


def localization_records(rows, maps, masks, threshold, tolerance):
    """Per-image peak hits and per-region detections for anomalous images."""
    images, regions = [], []
    for i, row in enumerate(rows):
        if not row['is_anomaly']:
            continue
        mask, heatmap = masks[i], maps[i]
        record = {'sample_id': row['sample_id'], 'group': row['group'], 'empty_mask': not mask.any(),
                  'peak_distance': None, 'hit_0px': None, 'hit_tol': None, 'n_regions': 0, 'n_detected': 0}
        if record['empty_mask']:
            images.append(record)
            continue
        peak = np.unravel_index(int(np.argmax(heatmap)), heatmap.shape)
        distance = float(ndimage.distance_transform_edt(~mask)[peak])
        labels, count = ndimage.label(mask, structure=STRUCTURE)
        sizes = np.bincount(labels.ravel(), minlength=count + 1)[1:]
        peaks = np.asarray(ndimage.maximum(heatmap, labels, index=np.arange(1, count + 1)), dtype=np.float64)
        detected = peaks >= threshold
        record.update(peak_distance=distance, hit_0px=distance == 0, hit_tol=distance <= tolerance,
                      n_regions=int(count), n_detected=int(detected.sum()))
        images.append(record)
        for size, hit in zip(sizes, detected):
            regions.append({'sample_id': row['sample_id'], 'group': row['group'], 'region_pixels': int(size),
                            'size_bin': size_bin(int(size)), 'detected': bool(hit)})
    return images, regions


def evaluate_extended(rows, scores, maps, masks, normal_fpr=.01, tolerance=8):
    if tolerance <= 0:
        raise ValueError('tolerance must be positive; the 0 px hit rate is always reported.')
    scores = np.asarray(scores, dtype=np.float64)
    maps, masks = np.asarray(maps), np.asarray(masks, dtype=bool)
    if maps.shape != masks.shape or maps.ndim != 3 or len(rows) != len(maps) or len(scores) != len(rows):
        raise ValueError('rows, scores, maps, and masks disagree in length or shape.')
    if not (np.isfinite(maps).all() and np.isfinite(scores).all()):
        raise ValueError('Non-finite scores or anomaly maps.')
    labels = np.array([r['is_anomaly'] for r in rows])
    groups = np.array([r['group'] for r in rows])
    threshold = normal_pixel_threshold(rows, maps, normal_fpr)
    images, regions = localization_records(rows, maps, masks, threshold, tolerance)
    names = metric_names(tolerance, normal_fpr)
    metrics = {}
    for group in GROUPS:
        selected = np.ones(len(rows), dtype=bool) if group == 'all' else ((groups == group) | (groups == 'normal'))
        positives = selected & (labels == 1)
        in_group = lambda r: group == 'all' or r['group'] == group
        group_images = [r for r in images if in_group(r)]
        valid = [r for r in group_images if not r['empty_mask']]
        group_regions = [r for r in regions if in_group(r)]
        entry = {'n_normal': int((selected & (labels == 0)).sum()), 'n_anomaly': int(positives.sum()),
                 'n_anomaly_empty_transformed_mask': sum(r['empty_mask'] for r in group_images),
                 'n_regions': len(group_regions), **{name: None for name in names}}
        if positives.any():
            current_masks, current_maps = masks[selected], maps[selected]
            curve = pro_curve(current_masks, current_maps)  # one sort serves every FPR limit
            for limit in FPR_LIMITS:
                entry[f'aupro@{limit:g}'] = aupro_from_curve(curve, limit)
            entry['pixel_ap'], entry['pixel_f1max_oracle'] = ranking_ap_f1(current_masks, current_maps)
            entry['image_auroc'] = safe_auroc(labels[selected], scores[selected])
            entry['image_ap'] = ranking_ap_f1(labels[selected], scores[selected])[0]
            entry[f'image_tpr@fpr{IMAGE_FPR:g}'] = tpr_at_fpr(labels[selected], scores[selected])
        if valid:
            entry['pointing@0px'] = float(np.mean([r['hit_0px'] for r in valid]))
            entry[f'pointing@{tolerance:g}px'] = float(np.mean([r['hit_tol'] for r in valid]))
        if group_regions:
            entry[f'region_recall@nfpr{normal_fpr:g}'] = float(np.mean([r['detected'] for r in group_regions]))
        metrics[group] = entry
    result = {'groups': metrics, 'normal_pixel_threshold': threshold, 'normal_fpr': normal_fpr, 'tolerance_px': tolerance}
    return result, images, regions


def paired_bootstrap(images_a, images_b, group, repeats=2000, seed=2026):
    """Stratified paired image bootstrap of B minus A for peak hits and region recall.

    Each model keeps its own fitted normal-image threshold, so the interval is
    conditional on both fitted models and describes test-image sampling only.
    """
    if [r['sample_id'] for r in images_a] != [r['sample_id'] for r in images_b]:
        raise ValueError('Runs have different anomalous images or ordering.')
    keep = [i for i, r in enumerate(images_a)
            if not r['empty_mask'] and (group == 'all' or r['group'] == group)]
    if len(keep) < 10:
        return {'status': 'insufficient_samples', 'n_images': len(keep)}
    a, b = [images_a[i] for i in keep], [images_b[i] for i in keep]
    if [r['n_regions'] for r in a] != [r['n_regions'] for r in b]:
        raise ValueError('Region counts differ; the runs do not share masks.')
    hit_a, hit_b = (np.array([r['hit_tol'] for r in x], dtype=float) for x in (a, b))
    det_a, det_b = (np.array([r['n_detected'] for r in x], dtype=float) for x in (a, b))
    n_regions = np.array([r['n_regions'] for r in a], dtype=float)
    image_groups = np.array([r['group'] for r in a])
    strata = [np.flatnonzero(image_groups == g) for g in ('small', 'large')]
    rng = np.random.default_rng(seed)
    pointing, recall = [], []
    for _ in range(repeats):
        idx = np.concatenate([rng.choice(s, len(s), replace=True) for s in strata if len(s)])
        pointing.append(hit_b[idx].mean() - hit_a[idx].mean())
        recall.append((det_b[idx].sum() - det_a[idx].sum()) / n_regions[idx].sum())
    return {'status': 'ok', 'n_images': len(keep),
            'pointing_diff': float(hit_b.mean() - hit_a.mean()),
            'pointing_ci95': np.quantile(pointing, [.025, .975]).tolist(),
            'region_recall_diff': float((det_b.sum() - det_a.sum()) / n_regions.sum()),
            'region_recall_ci95': np.quantile(recall, [.025, .975]).tolist()}


def load_run(folder):
    rows = json.loads((folder / 'samples.json').read_text())
    with np.load(folder / 'predictions.npz', allow_pickle=False) as data:
        if data['sample_ids'].tolist() != [r['sample_id'] for r in rows]:
            raise ValueError(f'Prediction/sample ordering mismatch: {folder}')
        return rows, data['scores'], data['maps'], data['masks']


def budget_of(config):
    return f"bank{config['bank_vectors_requested']}" if config['bank_vectors_requested'] else f"ratio{config['coreset_ratio']:g}"


def aggregate(rows, keys, metrics):
    buckets = defaultdict(list)
    for row in rows:
        buckets[tuple(row[k] for k in keys)].append(row)
    summary = []
    for key, values in sorted(buckets.items(), key=lambda item: tuple(map(str, item[0]))):
        seeds = [v['seed'] for v in values]
        if len(set(seeds)) != len(seeds):
            raise ValueError(f'Duplicate seeds in {key}; narrow --root to one study.')
        row = dict(zip(keys, key))
        row.update(n_seeds=len(values), seeds=','.join(map(str, sorted(seeds))),
                   systems=';'.join(sorted({v.get('system', '') for v in values})))
        for metric in metrics:
            data = [v[metric] for v in values if v[metric] is not None]
            row[metric + '_mean'] = float(np.mean(data)) if data else None
            row[metric + '_std'] = float(np.std(data, ddof=1)) if len(data) > 1 else None
        summary.append(row)
    return summary


def robustness(summary, names):
    """Check the report's claims under every metric, using seed means."""
    index = {(r['category'], r['image_size'], r['budget'], r['group'], r['debug'], r['layers']): r for r in summary}
    layer_rows, size_rows = [], []
    for cat, size, budget, group, debug in sorted({k[:5] for k in index}, key=lambda k: tuple(map(str, k))):
        for metric in names:
            cells = {l: index.get((cat, size, budget, group, debug, l)) for l in LAYER_ORDER}
            if any(c is None or c[metric + '_mean'] is None for c in cells.values()):
                continue
            means = {l: cells[l][metric + '_mean'] for l in LAYER_ORDER}
            stds = [cells[l][metric + '_std'] for l in LAYER_ORDER if cells[l][metric + '_std'] is not None]
            layer_rows.append({
                'category': cat, 'image_size': size, 'budget': budget, 'group': group, 'debug': debug, 'metric': metric,
                **means, 'ranking': ' > '.join(sorted(LAYER_ORDER, key=lambda l: -means[l])),
                'layer2_minus_fusion': means['layer2'] - means['fusion'],
                'layer2_minus_layer3': means['layer2'] - means['layer3'],
                'max_seed_std': max(stds) if stds else None,
                'layer2_ge_fusion': means['layer2'] >= means['fusion'],
                'layer2_gt_layer3': means['layer2'] > means['layer3'],
            })
    for cat, budget, group, debug, layer in sorted({(k[0], k[2], k[3], k[4], k[5]) for k in index}, key=lambda k: tuple(map(str, k))):
        low, high = index.get((cat, 224, budget, group, debug, layer)), index.get((cat, 320, budget, group, debug, layer))
        if low is None or high is None:
            continue
        for metric in names:
            if low[metric + '_mean'] is None or high[metric + '_mean'] is None:
                continue
            size_rows.append({'category': cat, 'layers': layer, 'budget': budget, 'group': group, 'debug': debug,
                              'metric': metric, 'size224': low[metric + '_mean'], 'size320': high[metric + '_mean'],
                              'gain_320_minus_224': high[metric + '_mean'] - low[metric + '_mean'],
                              'size320_gt_224': high[metric + '_mean'] > low[metric + '_mean']})
    claims = []
    for metric in names:
        for group in GROUPS:
            for claim, source, field in [('layer2 >= fusion', layer_rows, 'layer2_ge_fusion'),
                                         ('layer2 > layer3', layer_rows, 'layer2_gt_layer3'),
                                         ('320 > 224', size_rows, 'size320_gt_224')]:
                cases = [r for r in source if r['metric'] == metric and r['group'] == group]
                if cases:
                    holds = sum(bool(r[field]) for r in cases)
                    claims.append({'metric': metric, 'group': group, 'claim': claim, 'n_cases': len(cases),
                                   'n_holds': holds, 'holds_fraction': holds / len(cases)})
    return layer_rows, size_rows, claims


def plot_region_recall(summary, output):
    panels = sorted({(r['category'], r['image_size'], r['budget'], r['debug']) for r in summary}, key=lambda k: tuple(map(str, k)))
    for cat, size, budget, debug in panels:
        fig, ax = plt.subplots(figsize=(6, 4))
        counts = {}
        for offset, layer in zip((-.08, 0, .08), LAYER_ORDER):
            cells = [r for r in summary if (r['category'], r['image_size'], r['budget'], r['debug'], r['layers']) == (cat, size, budget, debug, layer)]
            cells = sorted(cells, key=lambda r: SIZE_LABELS.index(r['size_bin']))
            if not cells:
                continue
            x = [SIZE_LABELS.index(r['size_bin']) + offset for r in cells]
            ax.errorbar(x, [r['recall_mean'] for r in cells], yerr=[r['recall_std'] or 0 for r in cells],
                        marker='o', capsize=3, label=layer)
            counts.update({r['size_bin']: r['n_regions'] for r in cells})
        ax.set_xticks(range(len(SIZE_LABELS)))
        ax.set_xticklabels([f'{b}\n(n={counts.get(b, 0)})' for b in SIZE_LABELS])
        ax.set(xlabel='Defect region size (transformed-mask pixels)', ylabel='Region recall',
               title=f'{cat}, input {size}, {budget}{" (debug)" if debug else ""}', ylim=(-.03, 1.03))
        ax.legend()
        fig.tight_layout()
        fig.savefig(output / f'region_recall_{cat}_{size}_{budget}{"_debug" if debug else ""}.png', dpi=160)
        plt.close(fig)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('--root', type=Path, default=Path('results/runs'))
    p.add_argument('--output', type=Path, default=Path('results/extended'))
    p.add_argument('--normal-fpr', type=float, default=.01)
    p.add_argument('--tolerance', type=float, default=8)
    p.add_argument('--repeats', type=int, default=2000)
    p.add_argument('--include-debug', action='store_true')
    args = p.parse_args()
    names = metric_names(args.tolerance, args.normal_fpr)
    per_seed, region_rows, images_by_run, hashes = [], [], {}, {}
    for complete in sorted(args.root.rglob('complete.json')):
        folder = complete.parent
        config = json.loads((folder / 'config.json').read_text())
        if config['debug_subset'] and not args.include_debug:
            continue
        rows, scores, maps, masks = load_run(folder)
        result, images, regions = evaluate_extended(rows, scores, maps, masks, args.normal_fpr, args.tolerance)
        key = {'category': config['category'], 'image_size': config['image_size'], 'layers': config['layers'],
               'budget': budget_of(config), 'seed': config['seed'], 'debug': bool(config['debug_subset'])}
        provenance = {'dataset_sha256': config['dataset_sha256'], 'study_source_sha256': config['study_source_sha256'],
                      'system': config['system']}
        for group in GROUPS:
            per_seed.append({**key, 'group': group, **result['groups'][group],
                             'normal_pixel_threshold': result['normal_pixel_threshold'], **provenance})
        run_id = tuple(key.values())
        images_by_run[run_id] = images
        hashes[run_id] = config['dataset_sha256']
        bins = defaultdict(list)
        for region in regions:
            bins[region['size_bin']].append(region['detected'])
        for label in SIZE_LABELS:
            if bins[label]:
                region_rows.append({**key, 'size_bin': label, 'n_regions': len(bins[label]),
                                    'recall': float(np.mean(bins[label])), 'system': config['system']})
        print(f"{folder}: small AUPRO@0.05={result['groups']['small']['aupro@0.05']}", flush=True)
    if not per_seed:
        print('No completed runs found (debug runs need --include-debug).')
        return
    output = args.output
    output.mkdir(parents=True, exist_ok=True)
    summary = aggregate(per_seed, ['category', 'image_size', 'layers', 'budget', 'group', 'debug',
                                   'dataset_sha256', 'study_source_sha256'], names)
    region_summary = aggregate(region_rows, ['category', 'image_size', 'layers', 'budget', 'debug', 'size_bin'], ['recall'])
    for row in region_summary:
        row['n_regions'] = next(r['n_regions'] for r in region_rows
                                if all(r[k] == row[k] for k in ('category', 'image_size', 'layers', 'budget', 'debug', 'size_bin')))
    layer_rows, size_rows, claims = robustness(summary, names)
    tol = f'pointing@{args.tolerance:g}px'
    boot_rows = []
    for run_id, images in sorted(images_by_run.items(), key=lambda item: tuple(map(str, item[0]))):
        category, size, layers, budget, seed, debug = run_id
        partner = (category, size, 'fusion', budget, seed, debug)
        if layers != 'layer2' or partner not in images_by_run:
            continue
        if hashes[run_id] != hashes[partner]:
            raise ValueError(f'Dataset hashes differ for {run_id}; pairing is invalid.')
        for group in ('all', 'small'):
            b = paired_bootstrap(images, images_by_run[partner], group, args.repeats)
            boot_rows.append({'category': category, 'image_size': size, 'budget': budget, 'seed': seed, 'debug': debug,
                              'group': group, 'direction': 'fusion minus layer2', 'status': b['status'],
                              'n_images': b['n_images'], f'{tol}_diff': b.get('pointing_diff'),
                              f'{tol}_ci95': b.get('pointing_ci95'),
                              'region_recall_diff': b.get('region_recall_diff'),
                              'region_recall_ci95': b.get('region_recall_ci95')})
    write_csv(output / 'per_seed_extended.csv', per_seed)
    write_csv(output / 'mean_std_extended.csv', summary)
    write_csv(output / 'region_recall_by_size_per_seed.csv', region_rows)
    write_csv(output / 'region_recall_by_size.csv', region_summary)
    write_csv(output / 'robustness_layers.csv', layer_rows)
    write_csv(output / 'robustness_size.csv', size_rows)
    write_csv(output / 'claims.csv', claims)
    write_csv(output / 'bootstrap_layer2_vs_fusion.csv', boot_rows)
    write_json(output / 'extended_summary.json', {
        'settings': {'normal_fpr': args.normal_fpr, 'tolerance_px': args.tolerance, 'repeats': args.repeats,
                     'include_debug': args.include_debug, 'fpr_limits': list(FPR_LIMITS), 'image_fpr': IMAGE_FPR,
                     'size_bins': list(SIZE_LABELS)},
        'notes': ['pixel_f1max_oracle selects its threshold on test masks; use it only as a diagnostic.',
                  'Region recall thresholds come from normal test images only.',
                  'Claims compare three-seed means; check max_seed_std before reading small gaps.'],
        'claims': claims, 'bootstrap': boot_rows})
    plot_region_recall(region_summary, output)
    print(f'Wrote extended metrics for {len(per_seed) // len(GROUPS)} runs to {output}')
    for row in claims:
        if row['group'] == 'small':
            print(f"{row['metric']:<24} {row['claim']:<17} {row['n_holds']}/{row['n_cases']}")


if __name__ == '__main__':
    main()

In [ ]:
%%writefile /content/CV_final_project/tests/test_extended_metrics.py
"""Checks for the post-hoc extended metrics used by the Colab notebook."""
import unittest

import numpy as np
from sklearn.metrics import average_precision_score

from extended_metrics import (aupro_from_curve, evaluate_extended, normal_pixel_threshold,
                              paired_bootstrap, ranking_ap_f1, size_bin, tpr_at_fpr)
from study.metrics import aupro, pro_curve


def toy(size=16, n_normal=3):
    rows = [{'sample_id': f'good/{i}', 'is_anomaly': 0, 'group': 'normal'} for i in range(n_normal)]
    masks = [np.zeros((size, size), bool) for _ in range(n_normal)]
    small = np.zeros((size, size), bool)
    small[2:4, 2:4] = True
    large = np.zeros((size, size), bool)
    large[8:14, 8:14] = True
    large[0, 15] = True  # a second, separate region
    rows += [{'sample_id': 'bad/small', 'is_anomaly': 1, 'group': 'small'},
             {'sample_id': 'bad/large', 'is_anomaly': 1, 'group': 'large'}]
    masks += [small, large]
    return rows, np.stack(masks)


class ExtendedMetrics(unittest.TestCase):
    def test_pixel_ap_matches_sklearn_with_ties(self):
        rng = np.random.default_rng(0)
        labels = rng.random(500) < .1
        scores = np.round(rng.random(500) + labels * .3, 1)
        ap, _ = ranking_ap_f1(labels, scores)
        self.assertAlmostEqual(ap, average_precision_score(labels, scores), places=12)

    def test_f1_max_known_value(self):
        _, f1 = ranking_ap_f1([1, 0, 1, 0], [.9, .8, .7, .1])
        self.assertAlmostEqual(f1, .8)

    def test_single_class_is_undefined(self):
        self.assertEqual(ranking_ap_f1([0, 0], [.1, .2]), (None, None))
        self.assertIsNone(tpr_at_fpr([1, 1], [.1, .2]))

    def test_perfect_maps(self):
        rows, masks = toy()
        maps = masks.astype(np.float32)
        scores = np.array([r['is_anomaly'] for r in rows], float)
        result, _, _ = evaluate_extended(rows, scores, maps, masks)
        for group in ('all', 'small', 'large'):
            entry = result['groups'][group]
            for key in ('aupro@0.01', 'pixel_ap', 'pixel_f1max_oracle', 'image_auroc', 'image_ap',
                        'image_tpr@fpr0.05', 'pointing@0px', 'pointing@8px', 'region_recall@nfpr0.01'):
                self.assertAlmostEqual(entry[key], 1.0, msg=f'{group}/{key}')
        self.assertEqual(result['groups']['large']['n_regions'], 2)

    def test_inverted_maps_miss_every_region(self):
        rows, masks = toy()
        maps = 1 - masks.astype(np.float32)
        result, _, _ = evaluate_extended(rows, np.zeros(len(rows)), maps, masks)
        entry = result['groups']['all']
        self.assertEqual(entry['pointing@0px'], 0.0)
        self.assertEqual(entry['region_recall@nfpr0.01'], 0.0)

    def test_aupro_matches_study_implementation(self):
        rows, masks = toy()
        maps = np.random.default_rng(1).random(masks.shape).astype(np.float32)
        result, _, _ = evaluate_extended(rows, np.zeros(len(rows)), maps, masks)
        curve = pro_curve(masks, maps)
        for limit in (.3, .05, .01):
            self.assertAlmostEqual(result['groups']['all'][f'aupro@{limit:g}'], aupro(masks, maps, limit), places=12)
            self.assertAlmostEqual(aupro_from_curve(curve, limit), aupro(masks, maps, limit), places=12)

    def test_empty_transformed_mask_is_counted_not_scored(self):
        rows, masks = toy()
        rows.append({'sample_id': 'bad/vanished', 'is_anomaly': 1, 'group': 'small'})
        masks = np.concatenate([masks, np.zeros((1,) + masks.shape[1:], bool)])
        maps = masks.astype(np.float32)
        result, images, _ = evaluate_extended(rows, np.zeros(len(rows)), maps, masks)
        small = result['groups']['small']
        self.assertEqual(small['n_anomaly'], 2)
        self.assertEqual(small['n_anomaly_empty_transformed_mask'], 1)
        self.assertEqual(small['pointing@0px'], 1.0)
        self.assertTrue(images[-1]['empty_mask'])

    def test_size_bin_edges(self):
        expected = {1: '<64', 63: '<64', 64: '64-255', 255: '64-255', 256: '256-1023',
                    1023: '256-1023', 1024: '>=1024'}
        for pixels, label in expected.items():
            self.assertEqual(size_bin(pixels), label)

    def test_threshold_ignores_defect_images(self):
        rows, masks = toy()
        maps = np.random.default_rng(2).random(masks.shape)
        before = normal_pixel_threshold(rows, maps, .01)
        maps[[r['is_anomaly'] == 1 for r in rows]] *= 100
        self.assertEqual(before, normal_pixel_threshold(rows, maps, .01))

    def test_pointing_tolerance(self):
        rows = [{'sample_id': 'good', 'is_anomaly': 0, 'group': 'normal'},
                {'sample_id': 'bad', 'is_anomaly': 1, 'group': 'small'}]
        masks = np.zeros((2, 32, 32), bool)
        masks[1, 10, 10] = True
        maps = np.zeros((2, 32, 32))
        maps[1, 10, 15] = 1.0  # peak 5 px from the defect
        result, _, _ = evaluate_extended(rows, np.array([0., 1.]), maps, masks, tolerance=8)
        self.assertEqual(result['groups']['small']['pointing@0px'], 0.0)
        self.assertEqual(result['groups']['small']['pointing@8px'], 1.0)

    def test_tpr_at_low_fpr(self):
        labels = [0] * 20 + [1] * 5
        scores = list(range(20)) + [30, 31, 32, 33, 34]
        self.assertEqual(tpr_at_fpr(labels, scores, .05), 1.0)
        scores[0] = 40  # one normal image now outranks every defect
        self.assertEqual(tpr_at_fpr(labels, scores, .05), 1.0)  # FPR 1/20 is still allowed
        scores[1] = 41
        self.assertEqual(tpr_at_fpr(labels, scores, .05), 0.0)

    def test_bootstrap_of_identical_runs_is_zero(self):
        images = [{'sample_id': str(i), 'group': 'small' if i % 3 else 'large', 'empty_mask': False,
                   'hit_tol': i % 2 == 0, 'n_regions': 2, 'n_detected': i % 3} for i in range(12)]
        result = paired_bootstrap(images, images, 'all', repeats=200)
        self.assertEqual(result['status'], 'ok')
        self.assertEqual(result['pointing_diff'], 0.0)
        self.assertEqual(result['pointing_ci95'], [0.0, 0.0])
        self.assertEqual(result['region_recall_ci95'], [0.0, 0.0])

    def test_bootstrap_rejects_mismatched_runs(self):
        a = [{'sample_id': 'x', 'group': 'small', 'empty_mask': False, 'hit_tol': True, 'n_regions': 1, 'n_detected': 1}]
        b = [dict(a[0], sample_id='y')]
        with self.assertRaises(ValueError):
            paired_bootstrap(a, b, 'all')


if __name__ == '__main__':
    unittest.main()

In [ ]:
# 原有的 9 个科学性测试 + 扩展指标测试
sh([PY, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], cwd=REPO_DIR)

## 5. 下载 MVTec AD 数据与预训练权重

压缩包保存在 Drive（`archives/`），新会话只需重新解压，不必重新下载。数据来自 MVTec 官网链接（mydrive.ch），有时很慢；也可以先手动下载 `screw.tar.xz`、`capsule.tar.xz`、`metal_nut.tar.xz` 放进 `archives/`。数据集许可为 CC BY-NC-SA 4.0。

In [ ]:
sh([PY, 'download_data.py', '--archives', ARCHIVE_DIR, '--data', DATA_DIR], cwd=REPO_DIR)
sh([PY, 'prepare_weights.py'], cwd=REPO_DIR)  # ImageNet V1 WideResNet50-2 权重，并核对官方哈希前缀

## 6. 数据审计

统计每张缺陷图在原图中的面积占比（≤1% 为小缺陷），以及中心裁剪会不会裁掉缺陷像素。

In [ ]:
for size in (224, 320):
    sh([PY, 'inspect_dataset.py', '--data', DATA_DIR, '--size', size, '--output', OUT_DIR / f'audit/size{size}'], cwd=REPO_DIR)

## 7. 续跑工具

- `run_batch()` 依次运行一组实验。命令与 `run_matrix.py` 生成的完全相同；已有 `complete.json` 的运行会被直接跳过。
- `clean_incomplete()` 把没跑完的运行目录移到 `DRIVE_ROOT/incomplete/`。否则 `run_experiment.py` 会因为目录非空而报错停下。
- `progress()` 显示完成进度，并检查已完成的运行是否来自同一份数据和代码。

In [ ]:
CATEGORIES = ['screw', 'capsule', 'metal_nut']
LAYERS = ['layer2', 'layer3', 'fusion']
SEEDS = [0, 1, 2]
# 顺序与 run_matrix.py 相同：类别 → 尺寸 → 特征 → 种子
MAIN_JOBS = [dict(category=c, size=s, layers=l, seed=k, bank=None)
             for c in CATEGORIES for s in (224, 320) for l in LAYERS for k in SEEDS]
BANK_JOBS = [dict(category=c, size=224, layers=l, seed=k, bank=4096)
             for c in CATEGORIES for l in LAYERS for k in SEEDS]
MATRIX_LOG = LOG_DIR / 'matrix.log'


def run_folder(job, root=RUNS_DIR):
    tag = f"bank{job['bank']}" if job['bank'] else 'ratio0.1'
    return root / 'formal' / job['category'] / f"size{job['size']}" / job['layers'] / tag / f"seed{job['seed']}"


def is_complete(job):
    return (run_folder(job) / 'complete.json').exists()


def clean_incomplete():
    formal = RUNS_DIR / 'formal'
    if not formal.exists():
        return
    stamp = time.strftime('%Y%m%d-%H%M%S')
    for folder in sorted(formal.rglob('seed*')):
        if folder.is_dir() and any(folder.iterdir()) and not (folder / 'complete.json').exists():
            target = DRIVE_ROOT / 'incomplete' / stamp / folder.relative_to(RUNS_DIR)
            target.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(folder), str(target))
            print('已移走未完成的运行：', folder, '→', target)


def run_job(job):
    cmd = [PY, '-u', 'run_experiment.py', '--data', DATA_DIR, '--category', job['category'],
           '--size', job['size'], '--layers', job['layers'], '--seed', job['seed'],
           '--device', 'cuda', '--output', RUNS_DIR, '--resume']
    if job['bank']:
        cmd += ['--bank-vectors', job['bank']]
    sh(cmd, cwd=REPO_DIR, log=MATRIX_LOG)


def run_batch(jobs):
    clean_incomplete()
    todo = [j for j in jobs if not is_complete(j)]
    print(f'本批共 {len(jobs)} 组，其中 {len(jobs) - len(todo)} 组已完成，待运行 {len(todo)} 组。', flush=True)
    for i, job in enumerate(todo, 1):
        started = time.time()
        print(f"\n===== [{i}/{len(todo)}] {job} =====", flush=True)
        run_job(job)
        print(f'用时 {(time.time() - started) / 60:.1f} 分钟', flush=True)


def progress(show_missing=True):
    for name, jobs in (('主实验', MAIN_JOBS), ('固定记忆库实验', BANK_JOBS)):
        done = [j for j in jobs if is_complete(j)]
        print(f'{name}：{len(done)}/{len(jobs)} 已完成')
        if show_missing:
            for j in jobs:
                if not is_complete(j):
                    print('   未完成：', j)
    configs = [json.loads((run_folder(j) / 'config.json').read_text())
               for j in MAIN_JOBS + BANK_JOBS if is_complete(j)]
    if configs:
        print('源码哈希种类：', {c['study_source_sha256'][:12] for c in configs})
        print('系统种类：', {c['system'] for c in configs})
        for category in CATEGORIES:
            hashes = {c['dataset_sha256'][:12] for c in configs if c['category'] == category}
            if len(hashes) > 1:
                print(f'⚠️ {category} 的数据哈希不一致：{hashes}')

progress(show_missing=False)

## 8. 小规模试跑（可选，第一次运行时建议执行）

只用 20 张训练图、每种缺陷 2 张测试图，确认整个流程能走通，并大致感受速度。结果写到 Colab 本地的 `/content/pilot_runs/debug/`，不会混进正式结果。

In [ ]:
RUN_PILOT = True
if RUN_PILOT:
    PILOT_DIR = pathlib.Path('/content/pilot_runs')
    shutil.rmtree(PILOT_DIR, ignore_errors=True)
    started = time.time()
    sh([PY, '-u', 'run_experiment.py', '--data', DATA_DIR, '--category', 'screw', '--size', 320,
        '--layers', 'fusion', '--seed', 0, '--device', 'cuda', '--limit-train', 20,
        '--limit-test-per-type', 2, '--output', PILOT_DIR], cwd=REPO_DIR)
    print(f'试跑用时 {(time.time() - started) / 60:.1f} 分钟（正式运行的训练图约为 10 倍以上）')

## 9. 主实验（54 组）

按 类别 → 尺寸 → 特征 → 种子 的顺序运行，日志追加写到 `outputs/logs/matrix.log`。

- 一次会话跑不完也没关系：断线后从第 1 步起依次重新运行，到这里再执行本单元格，就会从断点继续。
- 想分批跑，可以设置 `ONLY_CATEGORIES` / `ONLY_SIZES`，例如 `ONLY_CATEGORIES = ['screw']`、`ONLY_SIZES = [224]`。

In [ ]:
ONLY_CATEGORIES = None  # 例如 ['screw']；None 表示全部
ONLY_SIZES = None       # 例如 [224]；None 表示全部

jobs = [j for j in MAIN_JOBS
        if (ONLY_CATEGORIES is None or j['category'] in ONLY_CATEGORIES)
        and (ONLY_SIZES is None or j['size'] in ONLY_SIZES)]
run_batch(jobs)
progress(show_missing=False)

## 10. 固定记忆库实验（27 组）

224 像素下，每个记忆库固定为 4096 个向量（16 MiB），在存储相同的条件下比较三种特征。

In [ ]:
ONLY_CATEGORIES = None  # 例如 ['capsule']

run_batch([j for j in BANK_JOBS if ONLY_CATEGORIES is None or j['category'] in ONLY_CATEGORIES])
progress(show_missing=False)

## 11. 进度检查

全部完成时应显示 54/54 和 27/27，源码哈希和系统最好各只有一种。

In [ ]:
progress()

## 12. 复现原报告的汇总与分析

1. `summarize_results.py`：三种子的均值和标准差，以及记忆库大小与 AUPRO 的权衡图
2. `compare_runs.py`：layer2 vs fusion 的配对分层 bootstrap（seed 0，3 类 × 2 尺寸 = 6 对），文件名与仓库里的 `results/comparisons/` 相同
3. `plot_comparison.py`：同一张小缺陷测试图在不同特征和尺寸下的热力图

In [ ]:
sh([PY, 'summarize_results.py', '--root', RUNS_DIR, '--output', OUT_DIR / 'summary_colab'], cwd=REPO_DIR)

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

rows = []
for category in CATEGORIES:
    for size in (224, 320):
        a = run_folder(dict(category=category, size=size, layers='layer2', seed=0, bank=None))
        b = run_folder(dict(category=category, size=size, layers='fusion', seed=0, bank=None))
        if not ((a / 'complete.json').exists() and (b / 'complete.json').exists()):
            print('跳过（运行未完成）：', category, size)
            continue
        out = OUT_DIR / 'comparisons' / f'{category}_{size}_layer2_vs_fusion_seed0.json'
        subprocess.run([PY, 'compare_runs.py', str(a), str(b), '--output', str(out)],
                       cwd=REPO_DIR, check=True, stdout=subprocess.DEVNULL)
        result = json.loads(out.read_text())
        for group, g in result['groups'].items():
            rows.append({'category': category, 'size': size, 'group': group, 'status': g.get('status'),
                         'AUPRO layer2': g.get('exact_aupro_a'), 'AUPRO fusion': g.get('exact_aupro_b'),
                         'fusion − layer2': g.get('exact_difference_b_minus_a'), '95% CI': g.get('ci95_b_minus_a')})
display(pd.DataFrame(rows))

In [ ]:
(OUT_DIR / 'figures').mkdir(parents=True, exist_ok=True)
for category in CATEGORIES:
    runs = [run_folder(dict(category=category, size=s, layers=l, seed=0, bank=None)) for s in (224, 320) for l in LAYERS]
    if not all((r / 'complete.json').exists() for r in runs):
        print('跳过（运行未完成）：', category)
        continue
    out = OUT_DIR / 'figures' / f'{category}_small_comparison.png'
    sh([PY, 'plot_comparison.py', '--runs', *runs, '--data', DATA_DIR, '--output', out], cwd=REPO_DIR)
    display(IPImage(filename=str(out)))

## 13. 扩展指标：用新指标检验原结论

原报告以 **AUPRO@0.3** 为主要指标，像素 AUROC 和图像 AUROC 为次要指标。下面的指标从不同角度检验同一批预测结果：

| 指标 | 含义 | 为什么有用 |
|---|---|---|
| AUPRO@0.05、AUPRO@0.01 | 只在很低的误报率区间内积分 PRO 曲线 | 实际质检只能容忍很少的误报；AUPRO@0.3 可能掩盖低误报区间的差异 |
| Pixel AP | 像素级 precision-recall 曲线下面积 | 小缺陷像素占比不到 1%，像素 AUROC 容易接近 1，AP 对这种极不平衡更敏感 |
| Image AP、Image TPR@FPR 5% | 图像级 AP；误报率 ≤5% 时能检出的缺陷图比例 | 图像级检测的另一个视角 |
| Pixel F1-max（oracle） | 所有阈值中最好的像素 F1 | 社区常用，但阈值是在测试掩码上挑的，**只作参考，不代表实际可达到的效果** |
| Pointing@0px / @8px | 异常图最大值像素是否落在缺陷内（或 8 像素以内） | 不需要阈值，直观反映模型能否"指到"小缺陷 |
| Region recall@nFPR 1% | 阈值取 normal 测试图像素分数的 99% 分位数；一个缺陷连通区域内的最大分数超过阈值即算检出 | 阈值不使用任何缺陷掩码；同时按区域大小分档，回答"多小的缺陷还能找到" |

主要输出（都在 `outputs/extended_colab/`）：
- `claims.csv`：每个指标下，三条结论各在多少个比较中成立
- `robustness_layers.csv` / `robustness_size.csv`：逐个比较的数值、排名和种子间标准差
- `bootstrap_layer2_vs_fusion.csv`：Pointing 和 Region recall 的配对 bootstrap 置信区间（每个种子单独计算）
- `region_recall_*.png`：区域大小与检出率的关系

In [ ]:
EXT_DIR = OUT_DIR / 'extended_colab'
sh([PY, 'extended_metrics.py', '--root', RUNS_DIR, '--output', EXT_DIR], cwd=REPO_DIR, log=LOG_DIR / 'extended.log')

In [ ]:
import pandas as pd
from IPython.display import Image as IPImage, display

pd.set_option('display.width', 200)
claims = pd.read_csv(EXT_DIR / 'claims.csv')
print('结论在各指标下成立的比例（small 组 = 全部 normal 图 + 小缺陷图）：')
table = claims[claims.group == 'small'].pivot_table(index='metric', columns='claim', values='holds_fraction')
display(table.style.format('{:.0%}').background_gradient(cmap='RdYlGn', vmin=0, vmax=1))

layers = pd.read_csv(EXT_DIR / 'robustness_layers.csv')
print('\nsmall 组、10% coreset：每个指标下三种特征的排名')
display(layers[(layers.group == 'small') & (layers.budget == 'ratio0.1')]
        [['category', 'image_size', 'metric', 'layer2', 'layer3', 'fusion', 'ranking', 'max_seed_std']])

boot = pd.read_csv(EXT_DIR / 'bootstrap_layer2_vs_fusion.csv')
print('\nlayer2 vs fusion 配对 bootstrap（正值表示 fusion 更好；区间包含 0 表示差异不显著）')
display(boot[boot.group == 'small'])

In [ ]:
for png in sorted(EXT_DIR.glob('region_recall_*.png')):
    display(IPImage(filename=str(png)))

## 14. 与原 Windows 结果对照

把 Colab 的逐种子结果和仓库里原 Windows 实验的 `results/summary_windows/per_seed.csv` 按 类别 / 尺寸 / 特征 / 种子 / 预算 / 分组 一一对应。

- 精度指标应当非常接近。GPU 不同会带来浮点差异，coreset 选点也可能略有不同，所以不会逐位一致。
- 延迟只并列显示，**不能比较**。

In [ ]:
import pandas as pd
from IPython.display import display

keys = ['category', 'image_size', 'layers', 'seed', 'budget', 'group']
win = pd.read_csv(REPO_DIR / 'results/summary_windows/per_seed.csv')
colab = pd.read_csv(OUT_DIR / 'summary_colab/per_seed.csv')
merged = win.merge(colab, on=keys, suffixes=('_win', '_colab'))
print(f'对应上的行数：{len(merged)}（81 组 × 3 个分组 = 243 时为完整）')
print('数据哈希全部一致：', bool((merged.dataset_sha256_win == merged.dataset_sha256_colab).all()))
print('源码哈希全部一致：', bool((merged.study_source_sha256_win == merged.study_source_sha256_colab).all()))

metrics = ['aupro', 'pixel_auroc', 'image_auroc']
for m in metrics:
    merged[f'{m}_diff'] = merged[f'{m}_colab'] - merged[f'{m}_win']
print('\n各指标 Colab − Windows 的最大绝对差：')
display(merged.groupby(['budget', 'group'])[[f'{m}_diff' for m in metrics]].agg(lambda s: s.abs().max()))

print('\nsmall 组 AUPRO@0.3 三种子均值：Windows vs Colab')
small = merged[merged.group == 'small'].groupby(['budget', 'category', 'image_size', 'layers'])[['aupro_win', 'aupro_colab']].mean()
display(small.unstack('layers').round(4))

print('\n延迟中位数（毫秒，三种子均值）——硬件不同，仅供参考，不可比较')
display(merged[merged.group == 'all'].groupby(['budget', 'category', 'image_size', 'layers'])
        [['median_latency_ms_win', 'median_latency_ms_colab']].mean().round(1))

## 15. 打包下载汇总结果

打包 `outputs/` 目录（审计、汇总、bootstrap、示例图、扩展指标、日志），不包括约 7.8 GB 的逐次运行输出（它们仍保存在 Drive 的 `runs_colab/` 中）。

In [ ]:
from google.colab import files

archive = shutil.make_archive('/content/group6_colab_outputs', 'zip', OUT_DIR)
print('已打包：', archive)
files.download(archive)